-------
## Edge Intelligence: Bandwidth Saving and Latency-Accuracy Trade-off
**Constants:**
| Constant              | Value / meaning |
|-----------------------|----------------|
| skip_interval         | 0 = process every frame |
| heuristic_threshold   | 5.0 = default heuristic change detector |
| video                 | same test video file for both runs (data/experiment_sample.mp4) |

**Edge-only run:**
To force all frames to be processed strictly on the edge I adjusted the following variable:
- `edge_conf_threshold = -1` (lower than the minimum possible), so `(max(confidences) < -1)` is always false → don’t send to the cloud.
- I amended the default logic which sent frames to the cloud automatically if the edge failed to detect anything, i.e., `if not confidences_list: dont send_to_cloud`, to keep the frame locally even if the edge failed to detect.

**Cloud-only run:**
To force the pipeline to send all claims to the cloud I implemented the following adjustments:
- `edge_conf_threshold = 1.1` (higher than the maximum possible), so `(max(confidences) < 1.1)` is always True → always send to the cloud.
- Maintained the default logic in sending frames to the cloud when the edge fails to detect objects (`if not confidences_list: send_to_cloud`).

----

In [1]:
from e_utilities import build_dataframe, bandwidth_bar_plot

----------
### CLoud-only

#### 1. Data

In [2]:
cloud_only = build_dataframe("../experiments/vms_cloud_only/metrics_history.json")

cloud_only.head()

,frame_index,timestamp,edge_inference_ms,avg_edge_inference_ms,total_frames_processed,heuristic_frames_dropped,heuristic_drop_ratio,cloud_avoidance_ratio,frames_sent_to_cloud,mb_sent_to_cloud,intrusion,alert_level,objects_count,frame_mean_conf,round_trip_time_ms,avg_rt_ms,cloud_infer_ms,avg_cloud_infer_ms
0,1,0.033333,68.956375,68.956375,2,1,0.500000,0.500000,1,0.052229,True,CRITICAL,8,0.598756,118.304968,118.304968,105.175972,105.175972
1,3,0.100000,17.838478,43.397427,4,2,0.500000,0.500000,2,0.104750,True,CRITICAL,7,0.637229,40.633202,79.469085,32.565832,68.870902
2,4,0.133333,11.793137,32.862663,5,2,0.400000,0.400000,3,0.157629,True,CRITICAL,7,0.613180,40.001869,66.313346,32.743216,56.828340
3,6,0.200000,13.326406,27.978599,7,3,0.428571,0.428571,4,0.210482,True,CRITICAL,7,0.643754,37.774801,59.178710,30.532598,50.254405
4,7,0.233333,17.143011,25.811481,8,3,0.375000,0.375000,5,0.263154,True,CRITICAL,7,0.619211,36.907196,54.724407,29.937983,46.191120


In [3]:
cloud_only.describe()

,frame_index,timestamp,edge_inference_ms,avg_edge_inference_ms,total_frames_processed,heuristic_frames_dropped,heuristic_drop_ratio,cloud_avoidance_ratio,frames_sent_to_cloud,mb_sent_to_cloud,objects_count,frame_mean_conf,round_trip_time_ms,avg_rt_ms,cloud_infer_ms,avg_cloud_infer_ms
count,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000
mean,2749.233099,91.641103,14.970595,13.821019,2750.233099,1610.733099,0.587987,0.587987,1139.500000,58.880521,7.406936,0.560616,37.066795,37.069407,30.243683,30.237082
std,1582.430047,52.747668,6.796142,1.584983,1582.430047,928.399455,0.043744,0.043744,657.746278,34.174740,3.367400,0.124159,3.155833,2.218997,2.886131,2.029939
min,1.000000,0.033333,10.868549,13.075461,2.000000,1.000000,0.250000,0.250000,1.000000,0.052229,0.000000,0.000000,31.985044,36.586106,26.200056,29.688256
25%,1399.250000,46.641667,12.183189,13.357217,1400.250000,830.000000,0.568341,0.568341,570.250000,28.986201,5.000000,0.480573,34.580767,36.793629,27.694404,30.006558
50%,2883.500000,96.116667,13.290167,13.491443,2884.500000,1745.000000,0.590346,0.590346,1139.500000,58.712386,7.000000,0.539634,37.368536,36.881280,30.562043,30.071223
75%,4271.500000,142.383333,14.848948,14.174896,4272.500000,2563.750000,0.601116,0.601116,1708.750000,88.799738,10.000000,0.614356,38.895309,37.005394,32.015920,30.189686
max,5107.000000,170.233333,68.956375,68.956375,5108.000000,2830.000000,0.855072,0.855072,2278.000000,117.750935,19.000000,0.967537,118.304968,118.304968,105.175972,105.175972


----

## 2. Final Snapshot Metrics

#### 2.1 Total frames processed

In [4]:
cloud_only[-1:]["total_frames_processed"]

2277    5108
Name: total_frames_processed, dtype: int64

#### 2.2 Frames sent to the cloud

In [5]:
cloud_only[-1:]["frames_sent_to_cloud"]

2277    2278
Name: frames_sent_to_cloud, dtype: int64

#### 2.3 Total data sent/ bandwidth usage (MB)

In [6]:
cloud_mb = cloud_only[-1:]["mb_sent_to_cloud"]

cloud_mb

2277    117.750935
Name: mb_sent_to_cloud, dtype: float64

------
#### 2.4 Heuristic filter drop ratio

In [7]:
cloud_only[-1:]["heuristic_drop_ratio"]

2277    0.554033
Name: heuristic_drop_ratio, dtype: float64

#### 2.5 Cloud avoidance ratio

In [8]:
cloud_only[-1:]["cloud_avoidance_ratio"]

2277    0.554033
Name: cloud_avoidance_ratio, dtype: float64

**Note**

This is due to the 5.0 default heuristic filter = heuristic drop ratio. Meaning all frames not dropped heuristically were send to the cloud for processing.

----

#### 2.6 Latency (avg round-trip)

In [9]:
cloud_only[-1:]["avg_rt_ms"]

2277    37.066795
Name: avg_rt_ms, dtype: float64

#### 2.7 Edge gateway avg infer time (ms)

In [10]:
cloud_only[-1:]["avg_edge_inference_ms"]

2277    14.970595
Name: avg_edge_inference_ms, dtype: float64

**Important Note**

Edge model serves as a decision gateway in this setup, the decision is always send to the cloud. This doesn't impact the round-trip latency of the cloud server, as the round-tip latency is measured after this gateway.

---------------------
-------

## Edge-only

#### 1. Data

In [11]:
edge_only_raw = build_dataframe("../experiments/vms_edge_only/metrics_history.json")
edge_only_raw.head()

,frame_index,timestamp,edge_inference_ms,avg_edge_inference_ms,total_frames_processed,heuristic_frames_dropped,heuristic_drop_ratio,cloud_avoidance_ratio,frames_sent_to_cloud,mb_sent_to_cloud,intrusion,alert_level,objects_count,frame_mean_conf,round_trip_time_ms,avg_rt_ms,cloud_infer_ms,avg_cloud_infer_ms
0,1,0.033333,75.465202,75.465202,2,1,0.500000,1.0,0,0.0,True,CRITICAL,9,0.490358,0.0,0.0,0.0,0.0
1,3,0.100000,20.239353,47.852278,4,2,0.500000,1.0,0,0.0,True,CRITICAL,7,0.565866,0.0,0.0,0.0,0.0
2,4,0.133333,12.104750,35.936435,5,2,0.400000,1.0,0,0.0,True,CRITICAL,7,0.589780,0.0,0.0,0.0,0.0
3,6,0.200000,12.522697,30.083001,7,3,0.428571,1.0,0,0.0,True,CRITICAL,8,0.586682,0.0,0.0,0.0,0.0
4,7,0.233333,11.076212,26.281643,8,3,0.375000,1.0,0,0.0,True,CRITICAL,8,0.583936,0.0,0.0,0.0,0.0


In [12]:
edge_only_raw.describe()

,frame_index,timestamp,edge_inference_ms,avg_edge_inference_ms,total_frames_processed,heuristic_frames_dropped,heuristic_drop_ratio,cloud_avoidance_ratio,frames_sent_to_cloud,mb_sent_to_cloud,objects_count,frame_mean_conf,round_trip_time_ms,avg_rt_ms,cloud_infer_ms,avg_cloud_infer_ms
count,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000,4556.000000
mean,2749.233099,91.641103,16.492439,16.010881,2750.233099,1610.733099,0.587987,0.999820,0.663740,0.031731,5.385206,0.530502,12.289909,12.690184,10.153103,10.454513
std,1582.256335,52.741878,9.232577,2.402022,1582.256335,928.297540,0.043740,0.000281,1.019981,0.048453,2.619146,0.138895,18.536033,19.096894,15.328944,15.733687
min,1.000000,0.033333,10.175467,13.119608,2.000000,1.000000,0.250000,0.998966,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1399.000000,46.633333,12.330353,14.528348,1400.000000,830.000000,0.568336,0.999550,0.000000,0.000000,4.000000,0.440544,0.000000,0.000000,0.000000,0.000000
50%,2883.500000,96.116667,13.611197,16.811133,2884.500000,1745.000000,0.590346,1.000000,0.000000,0.000000,5.000000,0.498534,0.000000,0.000000,0.000000,0.000000
75%,4272.000000,142.400000,15.496850,17.160547,4273.000000,2564.000000,0.601124,1.000000,2.000000,0.096952,7.000000,0.582996,39.081335,41.189909,32.117367,33.877492
max,5107.000000,170.233333,89.071274,89.071274,5108.000000,2830.000000,0.855072,1.000000,3.000000,0.136736,16.000000,0.968554,45.223236,43.298483,38.203955,35.637617


**Note:**

I conducted two runs for edge-only configuration. The first one was inaccurate as the `(if not confidences_list: send_to_cloud)` did send few frames to the cloud. I corrected it in the second run.

#### Drop the older run entries

In [13]:
edge_only_raw.drop(edge_only_raw.index[:2278], inplace=True)
# Drop old run logs
edge_only_raw.reset_index(drop=True, inplace=True)

edge_only = edge_only_raw

In [14]:
edge_only.head()

,frame_index,timestamp,edge_inference_ms,avg_edge_inference_ms,total_frames_processed,heuristic_frames_dropped,heuristic_drop_ratio,cloud_avoidance_ratio,frames_sent_to_cloud,mb_sent_to_cloud,intrusion,alert_level,objects_count,frame_mean_conf,round_trip_time_ms,avg_rt_ms,cloud_infer_ms,avg_cloud_infer_ms
0,1,0.033333,89.071274,89.071274,2,1,0.500000,1.0,0,0.0,True,CRITICAL,9,0.490358,0.0,0.0,0.0,0.0
1,3,0.100000,18.060207,53.565741,4,2,0.500000,1.0,0,0.0,True,CRITICAL,7,0.565866,0.0,0.0,0.0,0.0
2,4,0.133333,11.140585,39.424022,5,2,0.400000,1.0,0,0.0,True,CRITICAL,7,0.589780,0.0,0.0,0.0,0.0
3,6,0.200000,13.914585,33.046663,7,3,0.428571,1.0,0,0.0,True,CRITICAL,8,0.586682,0.0,0.0,0.0,0.0
4,7,0.233333,22.495508,30.936432,8,3,0.375000,1.0,0,0.0,True,CRITICAL,8,0.583936,0.0,0.0,0.0,0.0


In [15]:
edge_only.describe()

,frame_index,timestamp,edge_inference_ms,avg_edge_inference_ms,total_frames_processed,heuristic_frames_dropped,heuristic_drop_ratio,cloud_avoidance_ratio,frames_sent_to_cloud,mb_sent_to_cloud,objects_count,frame_mean_conf,round_trip_time_ms,avg_rt_ms,cloud_infer_ms,avg_cloud_infer_ms
count,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.000000,2278.0,2278.0,2278.0,2278.000000,2278.000000,2278.0,2278.0,2278.0,2278.0
mean,2749.233099,91.641103,17.512837,17.449706,2750.233099,1610.733099,0.587987,1.0,0.0,0.0,5.384548,0.530323,0.0,0.0,0.0,0.0
std,1582.430047,52.747668,10.432529,2.063411,1582.430047,928.399455,0.043744,0.0,0.0,0.0,2.620578,0.139311,0.0,0.0,0.0,0.0
min,1.000000,0.033333,11.022329,16.746128,2.000000,1.000000,0.250000,1.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
25%,1399.250000,46.641667,12.719274,16.952927,1400.250000,830.000000,0.568341,1.0,0.0,0.0,4.000000,0.440443,0.0,0.0,0.0,0.0
50%,2883.500000,96.116667,13.978720,17.131582,2884.500000,1745.000000,0.590346,1.0,0.0,0.0,5.000000,0.498516,0.0,0.0,0.0,0.0
75%,4271.500000,142.383333,15.836656,17.388508,4272.500000,2563.750000,0.601116,1.0,0.0,0.0,7.000000,0.582933,0.0,0.0,0.0,0.0
max,5107.000000,170.233333,89.071274,89.071274,5108.000000,2830.000000,0.855072,1.0,0.0,0.0,16.000000,0.968554,0.0,0.0,0.0,0.0


----
### 2. Final Snapshot Metrics

#### 2.1 Total frames processed

In [16]:
edge_only[-1:]["total_frames_processed"]

2277    5108
Name: total_frames_processed, dtype: int64

#### 2.2 Frames sent to the cloud

In [17]:
edge_only[-1:]["frames_sent_to_cloud"]

2277    0
Name: frames_sent_to_cloud, dtype: int64

#### 2.3 Total data sent/ bandwidth usage (MB)

In [18]:
edge_mb = edge_only[-1:]["mb_sent_to_cloud"]

edge_mb

2277    0.0
Name: mb_sent_to_cloud, dtype: float64

------
#### 2.4 Heuristic filter drop ratio

In [19]:
edge_only[-1:]["heuristic_drop_ratio"]

2277    0.554033
Name: heuristic_drop_ratio, dtype: float64

#### 2.5 Cloud avoidance ratio

In [20]:
edge_only[-1:]["cloud_avoidance_ratio"]

2277    1.0
Name: cloud_avoidance_ratio, dtype: float64

**Note**

Zero frames send to the cloud, all processed locally on-edge.

-----

#### 2.6 Latency (avg round-trip)

In [21]:
edge_only[-1:]["avg_rt_ms"]

2277    0.0
Name: avg_rt_ms, dtype: float64

#### 2.7 Average edge infer time latency

In [22]:
edge_only[-1:]["avg_edge_inference_ms"]

2277    17.512837
Name: avg_edge_inference_ms, dtype: float64

**Observation**

Zero RT latency as no frames were sent to the cloud. Edge average inference time is the only latency in this scenario.